# レッスン 08 - マルチエージェント設計パターン


## セットアップ


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## なぜマルチエージェントシステムなのか？

旅行計画のような実世界のタスクには、物流、地域の知識、予算管理など、多くの異なる専門知識が関わります。すべてを一人のエージェントが処理しようとすると、すぐに扱いにくくなります。

マルチエージェントシステムはこれを**専門化**によって解決します：各エージェントが一つの専門分野に特化することで、ジェネラリストよりも質の高い結果を生み出します。また、**スケーラビリティ**も向上します — 新しいエージェント（例えば、フライト専門家、レストラン批評家）を既存のワークフローを書き換えずに追加できます。エージェントは構造化されたパイプラインを通じて連携し、コンテキストを順に渡していきます。


## 専門エージェントの作成


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## 逐次ワークフローの構築

`WorkflowBuilder` を使うと、エージェントを有向グラフに接続できます。ここではシンプルな二段階のパイプラインを作成します：**TravelPlanner** が旅程を作成し、その後 **TravelConcierge** がそれを確認して充実させます。


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ワークフローへのエージェントの追加

マルチエージェントパターンの最大の利点の一つは、その拡張の容易さです。以下では、旅程を旅行者の予算と照らし合わせてチェックし、予算を超える可能性のある項目にフラグを立て、節約のための代替案を提案する**BudgetReviewer**エージェントを追加します。ワークフローは現在、3つのエージェントを順番に実行します：

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

このレッスンでは以下を学びました：

1. **専門化されたエージェントの作成** — 各エージェントは特定の役割（計画、コンシェルジュ、予算レビュー）を持ちます。
2. **`WorkflowBuilder` と `add_edge` を使ったエージェントの逐次的ワークフローへの接続**。
3. **マルチエージェントパイプラインからの出力をストリームし、どのエージェントが話しているかを追跡**。
4. **既存のエージェントを変更せずに、新しいエージェントをチェーンに追加してワークフローを拡張**。

マルチエージェント設計パターンは、各エージェントをシンプルに保ちながら、単一のエージェントだけでは達成できない、より豊かで綿密にレビューされた結果を生み出します。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：  
本書類はAI翻訳サービス[Co-op Translator](https://github.com/Azure/co-op-translator)を使用して翻訳されています。正確性には努めておりますが、自動翻訳には誤りや不正確な箇所が含まれる可能性があります。正式な情報源としては、原文を優先してください。重要な内容については、専門の人間による翻訳を推奨します。本翻訳の利用により生じた誤解や誤訳について、当社は一切の責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
